# Appendix: Magnitude distributions

The full photometric census is selected independently of spectroscopy. This notebook shows the selection difference that remains in the scored spectroscopic subset.

Run all cells in a Python environment with `requirements.txt` installed. The plotting style follows SpenderQ methods_paper and requires latex and dvipng. Inputs are frozen, documented snapshots in `results/figure_data/`; no training or network access is required. The final cell writes both PDF and PNG versions into `tex/figures/`.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'setup.py').exists())
sys.path.insert(0, str(ROOT / 'src'))
from pzflow.plotting import (COLORS, LABELS, set_style, load_predictions,
                         load_metrics, summary, bootstrap, save_figure)
set_style()

In [ ]:
with np.load(ROOT / 'results/figure_data/selection.npz') as f:
    d = {k: f[k] for k in f.files}
imag, split = d['imag'], d['split']
has_spec = np.isfinite(d['zspec'])
assert len(np.unique(d['id'])) == len(imag) == 162239
for s in range(3):
    for t in range(s):
        assert not np.intersect1d(d['cell'][split == s], d['cell'][split == t]).size
for label, mask in [('Full training', split == 0),
                    ('Spec training', (split == 0) & has_spec),
                    ('Photometric test', split == 2),
                    ('Spec test', (split == 2) & has_spec)]:
    print(f'{label}: N={mask.sum():,}; median i={np.median(imag[mask]):.3f}')

The magnitude is native HSC $i$ in a 2-arcsec aperture, before adding forecast noise. The spectroscopic samples preferentially contain brighter galaxies.

In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 2.8), layout='constrained')
bins = np.linspace(18, 25, 36)
groups = [(split == 0, 'Photometric training', COLORS['cosmos_pdf_flow']),
          ((split == 0) & has_spec, 'Broadband-flow training', COLORS['broadband_spec_flow']),
          ((split == 2) & has_spec, 'Spectroscopic test', '#333333')]
for mask, label, color in groups:
    vals = imag[mask]
    ax.hist(vals, bins=bins, weights=np.full(len(vals), 1 / len(vals) / .2),
            histtype='step', lw=1.35, color=color, label=f'{label} ({len(vals):,})')
ax.set(xlim=(18, 25), xlabel=r'HSC $i_{\mathrm{2\,arcsec}}$ [AB]', ylabel='Fraction per magnitude')
ax.legend(loc='upper left', frameon=False, fontsize=6.3)
save_figure(fig, ROOT, 'fig01_selection')
plt.show()
plt.close(fig)